# 03 — Modélisation
Entraînement et comparaison de 3 modèles supervisés en cross-validation 5 folds.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc

In [ ]:
df = pd.read_parquet('../data/twitter_clean.parquet')

FEATURES = [
    'statuses_count', 'followers_count', 'friends_count', 'favourites_count',
    'listed_count', 'default_profile', 'default_profile_image', 'geo_enabled',
    'profile_use_background_image', 'verified',
    'name_length', 'screen_name_length', 'description_length',
    'name_contains_bot', 'screen_name_contains_bot',
    'name_entropy', 'screen_name_entropy',
    'friends_followers_ratio', 'lists_followers_ratio',
    'retweet_followers_ratio', 'favorites_followers_ratio',
    'retweet_status_ratio', 'favorites_status_ratio', 'reply_status_ratio',
    'account_age',
    'followers_account_age_ratio', 'friends_account_age_ratio',
    'statuses_account_age_ratio', 'favourites_account_age_ratio',
    'lists_account_age_ratio',
]

X = df[FEATURES].fillna(0)
y = df['class_bot']
print(f'Dataset : {X.shape[0]} comptes, {X.shape[1]} features')
print(f'Bots : {y.sum()} ({y.mean()*100:.1f}%) | Humains : {(y==0).sum()}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## Définition des 3 modèles
Chaque modèle est encapsulé dans un Pipeline sklearn avec StandardScaler.

In [ ]:
models = {
    'Régression Logistique': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42)),
    ]),
    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('model', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)),
    ]),
}
print('Modèles définis :', list(models.keys()))

## Cross-validation 5 folds

In [ ]:
scoring = ['f1', 'precision', 'recall', 'roc_auc']
results = {}
for name, model in models.items():
    cv = cross_validate(model, X_train, y_train, cv=5, scoring=scoring)
    results[name] = {m: cv[f'test_{m}'].mean() for m in scoring}
    print(f'{name} — F1={results[name]["f1"]:.3f} | AUC={results[name]["roc_auc"]:.3f}')

In [ ]:
import pandas as pd
results_df = pd.DataFrame(results).T
results_df.plot.bar(figsize=(10, 5), ylim=(0.5, 1.05), color=['steelblue', 'seagreen', 'crimson', 'orange'])
plt.title('Comparaison CV 5 folds — 3 modèles')
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/comparaison_modeles.png', dpi=120)
plt.show()

## Entraînement final sur tout le train set

In [ ]:
model_keys = {'Régression Logistique': 'logistic_regression', 'Random Forest': 'random_forest', 'XGBoost': 'xgboost'}
for name, model in models.items():
    model.fit(X_train, y_train)
    path = f'../models/{model_keys[name]}.joblib'
    joblib.dump(model, path)
    print(f'Sauvegardé : {path}')

## Évaluation sur le jeu de test (20%)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f'{name} — F1={f1_score(y_test, y_pred):.3f} | Précision={precision_score(y_test, y_pred):.3f} | Rappel={recall_score(y_test, y_pred):.3f}')

## Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax,
        display_labels=['Humain', 'Bot'], cmap=plt.cm.Blues)
    ax.set_title(name)
plt.tight_layout()
plt.savefig('../plots/confusion_matrices.png', dpi=120)
plt.show()

## Courbes ROC

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in models.items():
    y_score = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('Faux positifs')
ax.set_ylabel('Vrais positifs')
ax.set_title('Courbes ROC')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/roc_curves.png', dpi=120)
plt.show()

## Feature importance — XGBoost

In [ ]:
import numpy as np
importances = models['XGBoost'].named_steps['model'].feature_importances_
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': importances}).sort_values('importance', ascending=True)

feat_df.plot.barh(x='feature', y='importance', figsize=(8, 8), color='steelblue', legend=False)
plt.title('Feature importance — XGBoost')
plt.xlabel('Importance (gain)')
plt.tight_layout()
plt.savefig('../plots/feature_importance.png', dpi=120)
plt.show()
print('Feature la plus importante :', feat_df.iloc[-1]['feature'])